# 10. LLM을 활용한 텍스트마이닝 기법

- 목표: LangChain과 OpenAI 모델을 이용해 요약, 키워드 추출, 감정 분류, 카테고리 분류, 임베딩 유사도 검색을 실습합니다.
- 사용 모델: `ChatOpenAI(model="gpt-4o-mini")`, `OpenAIEmbeddings(model="text-embedding-3-small")`
- 흐름: 프롬프트 작성 -> LLM 호출 -> 구조화된 결과 확인 -> 임베딩 벡터 유사도 계산


## 1. 실습 준비

이번 차시는 직접 모델을 학습하지 않고, 이미 학습된 LLM과 임베딩 모델을 텍스트마이닝 도구처럼 사용합니다.

필요한 패키지가 없다면 터미널에서 설치합니다.

```bash
pip install -U langchain langchain-openai langchain-core python-dotenv openai tiktoken
```

실습 폴더 루트에 `.env` 파일을 만들고 아래처럼 API 키를 저장합니다.

```bash
OPENAI_API_KEY=sk-...
```

API 키는 외부에 공개하면 안 됩니다. 노트북에는 키 값을 직접 적지 않고 `.env`에서 불러옵니다.


In [ ]:
from dotenv import load_dotenv  # .env 파일에 저장된 환경변수를 불러오는 함수입니다.

load_dotenv()  # 현재 폴더의 .env 파일을 읽어 OpenAI API 키를 사용할 수 있게 합니다.


### LangChain 기본 구성

LangChain에서는 보통 다음 세 가지를 연결해 사용합니다.

- `ChatPromptTemplate`: 모델에게 전달할 지시문과 입력 형식
- `ChatOpenAI`: OpenAI 채팅 모델 호출
- `StrOutputParser` 또는 `JsonOutputParser`: 응답을 문자열이나 JSON 형태로 정리

아래 코드는 이번 차시에서 반복해서 사용할 기본 객체를 준비합니다.


### 시스템 프롬프트란?

`ChatPromptTemplate.from_messages()`에서는 메시지를 역할별로 나눠 작성합니다.

- `system`: 모델의 기본 역할, 말투, 답변 기준을 정하는 지시문입니다.
- `user`: 실제 사용자가 요청하는 질문이나 분석 대상 텍스트입니다.
- `assistant`: 이전 답변 예시를 넣을 때 사용합니다. 이번 실습에서는 주로 `system`과 `user`만 사용합니다.

예를 들어 `("system", "너는 텍스트마이닝 분석가야.")`라고 쓰면, 이후 사용자 요청을 분석가 관점에서 처리하도록 방향을 잡아주는 역할을 합니다.  
수업에서는 시스템 프롬프트를 어렵게 생각하기보다 **모델에게 맡길 역할을 먼저 정하는 문장**으로 이해하면 됩니다.


In [ ]:
import json  # JSON 형태의 문자열과 파이썬 객체를 다룰 때 사용합니다.
import numpy as np  # 임베딩 벡터의 유사도 계산에 사용할 수치 연산 라이브러리입니다.
import pandas as pd  # 표 형태의 결과를 보기 좋게 정리하는 라이브러리입니다.

from langchain_openai import ChatOpenAI, OpenAIEmbeddings  # OpenAI 채팅 모델과 임베딩 모델을 LangChain에서 사용합니다.
from langchain_core.prompts import ChatPromptTemplate  # system/user 메시지로 프롬프트를 구성합니다.
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser  # 모델 응답을 문자열 또는 JSON으로 정리합니다.

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)  # 요약·분류 실습에 사용할 채팅 모델입니다. temperature=0은 답변을 비교적 일관되게 만듭니다.


## 2. 텍스트 요약

요약은 긴 리뷰, 뉴스, 회의록, 상담 기록에서 핵심 내용을 빠르게 파악할 때 자주 사용됩니다.  
요약 프롬프트에서는 **무엇을 남길지**, **얼마나 짧게 만들지**, **어떤 형식으로 출력할지**를 명확히 정합니다.


In [ ]:
sample_text = """
최근 한 달간 온라인 쇼핑몰의 생활가전 카테고리 리뷰 1,200건을 살펴보았다.
고객들은 대체로 제품 성능과 가격 대비 만족도에는 긍정적인 반응을 보였다.
특히 공기청정기와 무선청소기 상품에서는 흡입력, 소음 수준, 디자인에 대한 칭찬이 많았다.
배송 속도도 전반적으로 빠르다는 의견이 많았고, 지정일 배송을 편리하게 느낀 고객도 있었다.
다만 포장 상태에 대한 불만은 여러 상품군에서 반복적으로 나타났다.
일부 고객은 외부 박스가 찌그러져 도착했거나 완충재가 부족해 제품 파손을 걱정했다고 작성했다.
교환과 환불 절차에 대해서는 안내가 복잡하고 처리 상태를 확인하기 어렵다는 의견이 많았다.
고객센터 응답 속도도 주요 불만 요인으로 언급되었으며, 특히 주말 문의의 답변 지연이 자주 등장했다.
반면 상담원이 문제를 빠르게 해결해준 사례에서는 브랜드 신뢰도가 높아졌다는 평가도 있었다.
리뷰 작성자들은 제품 자체에는 만족하지만 구매 이후 문제가 생겼을 때의 지원 경험이 아쉽다고 정리했다.
재구매 의사가 있는 고객들도 포장 개선, 교환 절차 단순화, 배송 상태 알림 강화를 요구했다.
종합하면 상품 경쟁력은 유지되고 있으나, 물류와 사후 지원 경험을 개선하면 고객 만족도를 더 높일 수 있다.
""".strip()  # 요약과 키워드 추출에 사용할 긴 예시 텍스트입니다.

summary_prompt = ChatPromptTemplate.from_messages([  # 역할별 메시지로 프롬프트를 구성합니다.
    ("system", "너는 텍스트마이닝 분석가야."),  # 모델이 어떤 관점으로 답할지 정합니다.
    ("user", "다음 텍스트를 3문장 이내로 요약해줘.\n{text}")  # 실제 분석 요청과 입력 텍스트 자리입니다.
])

summary_chain = summary_prompt | llm | StrOutputParser()  # 프롬프트 -> 모델 -> 문자열 출력 순서로 체인을 만듭니다.
summary = summary_chain.invoke({"text": sample_text})  # {text} 자리에 sample_text를 넣어 체인을 실행합니다.
print(summary)  # 요약 결과를 출력합니다.


### 문제 1. 요약 관점 바꾸기

위 요약 프롬프트를 수정해 다음 형식으로 출력해 보세요.

- 핵심 긍정 의견
- 핵심 부정 의견
- 개선 제안

<details>
<summary>힌트 보기</summary>

`user` 메시지에 원하는 출력 항목을 직접 적으면 됩니다.

```python
("user", "긍정 의견, 부정 의견, 개선 제안으로 나눠 요약해줘.\n{text}")
```
</details>


## 3. 키워드 추출

키워드 추출은 문서에서 중요한 단어와 짧은 구를 뽑아 이슈를 요약하는 작업입니다.  
LLM을 사용할 때는 결과를 표로 다루기 쉽도록 JSON 형식으로 받는 것이 편합니다.


In [ ]:
keyword_parser = JsonOutputParser()  # 모델 응답을 파이썬 dict/list로 바꿔주는 JSON 파서입니다.

keyword_prompt = ChatPromptTemplate.from_messages([  # 키워드 추출용 프롬프트를 구성합니다.
    ("system", "너는 한국어 텍스트마이닝 분석가야."),  # 모델의 역할을 지정합니다.
    ("user", "키워드 5개를 JSON으로 추출해줘. 형식: {{\"keywords\": [{{\"keyword\": \"...\", \"reason\": \"...\"}}]}}\n{text}")  # 출력 형식과 분석할 텍스트를 전달합니다.
])

keyword_chain = keyword_prompt | llm | keyword_parser  # 프롬프트 -> 모델 -> JSON 파서 순서로 연결합니다.
keyword_result = keyword_chain.invoke({"text": sample_text})  # 예시 텍스트에서 키워드를 추출합니다.
keyword_result  # 추출된 JSON 결과를 확인합니다.


In [ ]:
keyword_df = pd.DataFrame(keyword_result["keywords"])  # keywords 리스트를 표 형태로 변환합니다.
keyword_df  # 키워드와 추출 이유를 데이터프레임으로 확인합니다.


## 4. 감정 분류

감정 분류는 리뷰나 댓글을 긍정, 부정, 중립으로 나누는 작업입니다.  
LLM은 문장의 표현을 읽고 분류할 수 있지만, 결과 기준을 명확히 적어야 일관성이 좋아집니다.


In [ ]:
review_df = pd.DataFrame({  # 감정 분류에 사용할 짧은 리뷰 예시를 만듭니다.
    "text": [
        "배송이 정말 빨라서 만족합니다.",
        "제품은 괜찮지만 포장이 너무 허술했어요.",
        "교환 신청을 했는데 답변이 너무 늦습니다.",
        "가격 대비 품질이 좋아서 재구매하고 싶어요.",
        "아직 사용 전이라 평가는 어렵습니다.",
    ]
})
review_df  # 분류 대상 리뷰 목록을 확인합니다.


In [ ]:
sentiment_prompt = ChatPromptTemplate.from_messages([  # 감정 분류용 프롬프트를 구성합니다.
    ("system", "너는 고객 리뷰를 분석하는 분류기야."),  # 모델의 역할을 분류기로 고정합니다.
    ("user", "감정을 positive, negative, neutral 중 하나로 JSON 분류해줘. 형식: {{\"sentiment\": \"...\", \"reason\": \"...\"}}\n리뷰: {text}")  # 리뷰 1건과 원하는 JSON 형식을 전달합니다.
])

sentiment_chain = sentiment_prompt | llm | JsonOutputParser()  # 프롬프트, 모델, JSON 파서를 하나의 체인으로 연결합니다.

sentiment_rows = []  # 리뷰별 분류 결과를 저장할 리스트입니다.
for text in review_df["text"]:  # 리뷰를 한 건씩 꺼내 모델에 전달합니다.
    sentiment_rows.append(sentiment_chain.invoke({"text": text}))  # 각 리뷰의 감정과 근거를 저장합니다.

sentiment_df = pd.concat([review_df, pd.DataFrame(sentiment_rows)], axis=1)  # 원본 리뷰와 분류 결과를 옆으로 붙입니다.
sentiment_df  # 최종 감정 분류 결과를 확인합니다.


## 5. 카테고리 분류

카테고리 분류는 상담 문의, 뉴스 기사, VOC를 미리 정한 업무 분류로 나누는 작업입니다.  
프롬프트에 가능한 카테고리 목록을 넣으면, 분류 기준을 더 안정적으로 만들 수 있습니다.


In [ ]:
inquiry_df = pd.DataFrame({  # 카테고리 분류에 사용할 고객 문의 예시를 만듭니다.
    "text": [
        "주문한 상품이 아직 도착하지 않았습니다.",
        "불량 제품을 받아서 환불하고 싶습니다.",
        "쿠폰 적용이 안 되는데 확인 부탁드립니다.",
        "제품 사용 중 전원이 자꾸 꺼집니다.",
        "회원 탈퇴 메뉴가 어디 있는지 모르겠습니다.",
    ]
})

categories = ["배송", "환불/교환", "가격/쿠폰", "상품품질", "계정/기타"]  # 모델이 선택할 수 있는 카테고리 목록입니다.
inquiry_df  # 분류 대상 문의 목록을 확인합니다.


In [ ]:
category_prompt = ChatPromptTemplate.from_messages([  # 카테고리 분류용 프롬프트를 구성합니다.
    ("system", "너는 고객 문의를 업무 카테고리로 분류하는 분석가야."),  # 모델의 역할을 업무 분류 분석가로 지정합니다.
    ("user", "카테고리 중 하나로 JSON 분류해줘. 형식: {{\"category\": \"...\", \"reason\": \"...\"}}\n카테고리: {categories}\n문의: {text}")  # 선택 가능한 카테고리와 문의 1건을 전달합니다.
])

category_chain = category_prompt | llm | JsonOutputParser()  # 프롬프트 -> 모델 -> JSON 파서 체인을 만듭니다.

category_rows = []  # 문의별 분류 결과를 저장할 리스트입니다.
for text in inquiry_df["text"]:  # 문의를 한 건씩 분류합니다.
    category_rows.append(category_chain.invoke({
        "categories": ", ".join(categories),  # 카테고리 리스트를 쉼표로 이어진 문자열로 전달합니다.
        "text": text,  # 현재 분류할 문의 문장입니다.
    }))

category_df = pd.concat([inquiry_df, pd.DataFrame(category_rows)], axis=1)  # 원본 문의와 분류 결과를 합칩니다.
category_df  # 최종 카테고리 분류 결과를 확인합니다.


## 6. OpenAI 임베딩으로 문서 유사도 계산

9번 파일에서는 텍스트를 벡터로 바꾸고 벡터 간 유사도를 계산하는 개념을 배웠습니다.  
이번에는 OpenAI 임베딩 모델로 문장을 벡터로 바꾼 뒤, 코사인 유사도를 직접 계산합니다.

이 방식은 검색, 추천, 중복 문서 탐지, RAG의 검색 단계에서 핵심적으로 사용됩니다.


In [ ]:
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")  # 문장을 임베딩 벡터로 바꿀 모델입니다.

text_docs = [  # 검색 대상이 되는 문서 목록입니다.
    "배송이 지연되어 고객 불만이 증가하고 있다.",
    "포장재가 약해 상품 파손 사례가 보고되었다.",
    "가격 할인 이벤트 이후 신규 고객 유입이 늘었다.",
    "환불 절차가 복잡해서 고객센터 문의가 많아졌다.",
    "제품 품질 만족도가 높아 재구매 의사가 증가했다.",
]

query = "교환과 환불 과정이 불편하다는 고객 의견"  # 검색 질문 또는 사용자의 관심 문장입니다.

doc_vectors = embedding_model.embed_documents(text_docs)  # 문서 목록을 각각 임베딩 벡터로 변환합니다.
query_vector = embedding_model.embed_query(query)  # 검색 문장도 같은 임베딩 공간의 벡터로 변환합니다.

print("문서 개수:", len(doc_vectors))  # 생성된 문서 벡터 개수를 확인합니다.
print("임베딩 차원:", len(query_vector))  # 임베딩 벡터의 길이를 확인합니다.
print("쿼리 벡터 앞 5개 값:", query_vector[:5])  # 벡터가 실제 숫자 배열인지 일부 값을 확인합니다.


In [ ]:
def cosine_similarity(vec_a, vec_b):  # 두 벡터의 코사인 유사도를 계산하는 함수입니다.
    a = np.array(vec_a)  # 첫 번째 벡터를 numpy 배열로 바꿉니다.
    b = np.array(vec_b)  # 두 번째 벡터를 numpy 배열로 바꿉니다.
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))  # 내적을 벡터 크기로 나눠 유사도를 계산합니다.

similarities = [cosine_similarity(query_vector, doc_vector) for doc_vector in doc_vectors]  # 쿼리와 각 문서의 유사도를 계산합니다.

similarity_df = pd.DataFrame({  # 문서별 유사도 결과를 표로 정리합니다.
    "query": query,
    "document": text_docs,
    "similarity": similarities,
}).sort_values("similarity", ascending=False)  # 유사도가 높은 문서부터 정렬합니다.

similarity_df  # 검색 결과처럼 가장 관련 높은 문서를 확인합니다.


### 문제 2. 나만의 임베딩 검색 만들기

아래 중 하나를 바꿔 실행해 보세요.

- `query`를 `배송 문제를 해결하고 싶다`로 변경
- `text_docs`에 새로운 문장 2개 추가
- 유사도가 가장 높은 문서 3개만 출력

<details>
<summary>힌트 보기</summary>

```python
similarity_df.head(3)
```
</details>


## 체크포인트

- LLM은 요약, 키워드 추출, 감정 분류, 카테고리 분류처럼 사람이 기준을 읽고 판단하던 작업을 빠르게 자동화할 수 있습니다.
- 출력 형식을 JSON으로 고정하면 결과를 데이터프레임으로 정리하기 쉽습니다.
- OpenAI 임베딩은 문장을 의미 벡터로 바꾸며, 코사인 유사도로 문서 검색과 추천을 구성할 수 있습니다.
- LLM 결과는 항상 검토가 필요합니다. 중요한 업무에서는 샘플 검수, 기준 문서화, 재현성 확인을 함께 해야 합니다.
